In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.10.0+cu128
12.8
True
Tesla T4


In [3]:
!pip install -q -U safetensors
!pip install -q -U transformers datasets accelerate safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.2 MB/s eta 0:00:00


In [4]:
import torch
from dataclasses import dataclass
from typing import Optional, Union

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
train_data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_data.shape, test_data.shape)
train_data.head()
# checking nulls just to be safe, dont wanna deal with NaN errors mid training
train_data.isnull().sum()
train_data['answer'].value_counts()
# looks fairly balanced across A-E, good

(2000, 8) (500, 7)


answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train_data, test_size=0.1, random_state=42, stratify=train_data['answer'])
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

len(train_split), len(val_split)

(1800, 200)

In [7]:
MODEL_NAME = "microsoft/deberta-v3-base"  # went with base first, can try large later if time permits
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [8]:
opts = ["A", "B", "C", "D", "E"]
ans_to_idx = {o: i for i, o in enumerate(opts)}

def make_labels(df):
    df = df.copy()
    df["label"] = df["answer"].map(ans_to_idx)
    return df

# tokenizer wants flat lists not nested ones, so squashing the 5 options per
# question into one long list, tokenizing, then splitting back into groups of 5
def tokenize_fn(examples):
    first_sent = [[q]*5 for q in examples["prompt"]]
    second_sent = []
    for i in range(len(examples["prompt"])):
        second_sent.append([examples[o][i] for o in opts])

    first_sent = sum(first_sent, [])
    second_sent = sum(second_sent, [])

    out = tok(first_sent, second_sent, truncation=True, max_length=256)
    out = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in out.items()}
    return out

In [9]:
train_split = make_labels(train_split)
val_split = make_labels(val_split)

train_ds = Dataset.from_pandas(train_split)
val_ds = Dataset.from_pandas(val_split)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

# quick sanity check, should be 5 sets of input_ids per row
len(train_ds[0]['input_ids'])

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

5

In [10]:
# HF doesn't have a ready made collator for multi choice, writing one myself
# basically after this the batch shape becomes (batch_size, num_choices, seq_len)
# which is what AutoModelForMultipleChoice expects as input
@dataclass
class MCQCollator:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str] = True

    def __call__(self, features):
        labels = [f.pop("label") for f in features]
        bs = len(features)
        n_choices = len(features[0]["input_ids"])

        flat = []
        for f in features:
            for i in range(n_choices):
                flat.append({k: v[i] for k, v in f.items()})

        batch = self.tokenizer.pad(flat, padding=self.padding, return_tensors="pt")
        batch = {k: v.view(bs, n_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

In [11]:
# writing this as a plain loop instead of vectorized, easier to explain step by step
def map_at_3(logits, labels):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    score = 0.0
    for pred_row, true_label in zip(top3, labels):
        if pred_row[0] == true_label:
            score += 1
        elif pred_row[1] == true_label:
            score += 1/2
        elif pred_row[2] == true_label:
            score += 1/3
        # else add 0, correct answer wasnt even in top 3
    return score / len(labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"map3": map_at_3(logits, labels), "acc": (preds == labels).mean()}

print('done')

done


In [12]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

# started with lr=2e-5 and it was kind of unstable, dropped it down
train_args = TrainingArguments(
    output_dir="mcq_out",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    fp16=False,
    report_to="none",
    logging_steps=25,
)

data_collator = MCQCollator(tokenizer=tok)

# newer transformers versions renamed "tokenizer" to "processing_class" in Trainer
# got a TypeError on tokenizer= first, this handles both cases so it just works
try:
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

trainer.train()

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight 

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=678, training_loss=3.8737373464578724, metrics={'train_runtime': 347.1466, 'train_samples_per_second': 31.111, 'train_steps_per_second': 1.953, 'total_flos': 2765959183541520.0, 'train_loss': 3.8737373464578724, 'epoch': 6.0})

In [13]:
test_copy = test_data.copy()
test_copy["label"] = 0  # dummy, test set has no real answer col, just need this so tokenize_fn doesnt break

test_ds = Dataset.from_pandas(test_copy)
test_ds = test_ds.map(tokenize_fn, batched=True)

result = trainer.predict(test_ds)
logits = result.predictions

idx_to_opt = {i: o for i, o in enumerate(opts)}
top3 = np.argsort(-logits, axis=1)[:, :3]
final_preds = [" ".join(idx_to_opt[i] for i in row) for row in top3]

sub = pd.DataFrame({"ID": test_data["id"], "Prediction": final_preds})
sub.to_csv("submission.csv", index=False)
sub.head()
print('done')

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


done


In [14]:
print(train_data['answer'].unique())

['B' 'A' 'C' 'E' 'D']


In [15]:
row = train_split.iloc[0]
print(row['prompt'], row['answer'], row['label'])
print(train_ds[0]['input_ids'][row['label']][:20])  # should correspond to correct option

What is the relevant type of coherence for the Young's double-slit interferometer? E 4
[1, 458, 269, 262, 2193, 810, 265, 33021, 270, 262, 3546, 280, 268, 1664, 271, 268, 13092, 104741, 302, 2]


In [16]:
print(trainer.evaluate())

Training Loss,Validation Loss,Epoch,Map3,Acc
0.000000,nan,6,0.384167,0.185000


{'eval_loss': nan, 'eval_map3': 0.38416666666666666, 'eval_acc': 0.185}


In [17]:
# make sure nothing shuffled between test_data and predictions
print(test_data['id'].head())
print(sub['ID'].head())
# confirm option column order matches what the model was trained on
print(opts)
print(train_data.columns.tolist())

0    1
1    2
2    3
3    4
4    5
Name: id, dtype: int64
0    1
1    2
2    3
3    4
4    5
Name: ID, dtype: int64
['A', 'B', 'C', 'D', 'E']
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
